## Install Dependencies

In [2]:
#Import anthropic
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 5.4 MB/s eta 0:00:00




---



# Agentic Loops with Real Use Case

### **Scenario 1: The Smart Logistic Assistant**

Our shipment of 500 sensors is delayed. Find the tracking number, check the new ETA, and if it's arriving after Friday, email the floor manager to pause production.


## **Why an Agentic Loop is a better approach**

- Compared to "one-shot" call cannot email the manager until the ETA

- Wihtout a loop the model would have to guess the outcome or stop halfway

- Reasoning → Action → Observation cycle



---



# Test API Connection is working

In [11]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# This is the correct ID format for the current April 2026 stable release
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=100,
    messages=[{"role": "user", "content": "Confirm connection. Give me a 2026 high-five!"}]
)

print(f"Claude says: {response.content[0].text}")

Claude says: **Connection confirmed! 🙌**

Here's your 2026 high-five:

**✋ H - Hopeful**
**✋ I - Innovative**
**✋ G - Grateful**
**✋ H - Hardy**
**✋ 5 - Five years into the decade and still going!**

Though I should be straightforward with you - I'm an AI, and my knowledge has a


In [13]:
import anthropic
from google.colab import userdata
import json

# 1. Setup the Secure Client
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# 2. Define the "Real World" Mock Functions
def get_order_info(item):
    """Simulates looking up an order database."""
    orders = {"sensors": "TRK-9904", "actuators": "TRK-2210"}
    tracking = orders.get(item.lower(), "NOT_FOUND")
    print(f"   [Tool Log] get_order_info: Found {tracking} for {item}")
    return {"tracking_number": tracking}

def check_shipping(tracking_number):
    """Simulates a carrier API. Returns an ETA date."""
    # Monday, April 20, 2026 is after Friday, April 17.
    print(f"   [Tool Log] check_shipping: Checking {tracking_number}...")
    return {"eta_date": "2026-04-20", "status": "Delayed"}

def send_email(recipient, body):
    """Simulates sending an email."""
    print(f"   [Tool Log] send_email: Sending to {recipient}...")
    print(f"   [Message Preview]: {body}")
    return {"status": "success", "confirmed_at": "11:35 AM"}

# 3. Tool Definitions for Claude
tools = [
    {
        "name": "get_order_info",
        "description": "Get the tracking number for a specific item shipment.",
        "input_schema": {
            "type": "object",
            "properties": {"item": {"type": "string"}},
            "required": ["item"]
        }
    },
    {
        "name": "check_shipping",
        "description": "Check the ETA for a tracking number.",
        "input_schema": {
            "type": "object",
            "properties": {"tracking_number": {"type": "string"}},
            "required": ["tracking_number"]
        }
    },
    {
        "name": "send_email",
        "description": "Send a notification email to a manager.",
        "input_schema": {
            "type": "object",
            "properties": {
                "recipient": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["recipient", "body"]
        }
    }
]

# 4. The Orchestration
messages = [{
    "role": "user",
    "content": "Our shipment of sensors is delayed. Find the tracking info, check the ETA, and if it's after Friday, email the floor manager to pause production."
}]

print("--- AGENT ACTIVATED (Model: claude-sonnet-4-6) ---\n")

while True:
    # Use the current 2026 stable model
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4096,
        tools=tools,
        messages=messages
    )

    # Termination Condition: Claude has finished the task
    if response.stop_reason == "end_turn":
        print("\n--- FINAL AGENT RESPONSE ---")
        print(response.content[0].text)
        break

    # Action Phase: Claude wants to use a tool
    if response.stop_reason == "tool_use":
        # Extract the tool block
        tool_block = next(b for b in response.content if b.type == "tool_use")
        tool_name = tool_block.name
        tool_input = tool_block.input
        tool_id = tool_block.id

        # Dispatch the call to our local Python functions
        if tool_name == "get_order_info":
            result = get_order_info(tool_input['item'])
        elif tool_name == "check_shipping":
            result = check_shipping(tool_input['tracking_number'])
        elif tool_name == "send_email":
            result = send_email(tool_input['recipient'], tool_input['body'])
        else:
            result = {"error": "Unknown tool"}

        # Memory Update: Keep the conversation chain alive
        # 1. Append the assistant's request (including the tool_use block)
        messages.append({"role": "assistant", "content": response.content})

        # 2. Append the tool result for the assistant to observe
        messages.append({
            "role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": tool_id,
                "content": json.dumps(result)
            }]
        })

--- AGENT ACTIVATED (Model: claude-sonnet-4-6) ---

   [Tool Log] get_order_info: Found TRK-9904 for sensors
   [Tool Log] check_shipping: Checking TRK-9904...
   [Tool Log] send_email: Sending to floor manager...
   [Message Preview]: Hi, I wanted to let you know that our sensors shipment (Tracking #: TRK-9904) has been delayed. The new estimated delivery date is April 20, 2026 (Monday), which is after Friday. Please pause production until the shipment arrives. Thank you.

--- FINAL AGENT RESPONSE ---
Here's a full summary of what was done:

1. 📦 **Tracking Info Found** — The sensors shipment is tracked under **TRK-9904**, with a status of **Delayed**.
2. 📅 **ETA Checked** — The shipment is now expected to arrive on **Monday, April 20, 2026**, which is past Friday.
3. 📧 **Email Sent** — The floor manager was notified at **11:35 AM** to **pause production** until the sensors arrive.

Let me know if there's anything else you'd like to do!


**Insights**

- The agent successfully processed a complex request requiring three distinct tool interactions. The logic flow was as follows:

- Entity Resolution: Mapped the ambiguous term "sensors" to the specific tracking ID TRK-9904.

- Dependency Chaining: Successfully used the output of Tool 1 as the required input for Tool 2 (check_shipping).

- Logic Gate Adherence: Verified that the ETA (Monday, April 20) fell after the defined threshold (Friday).

- Conditional Execution: Only triggered the side-effect tool (send_email) after the logic gate condition returned True.



---



**Business Impact**

- This test proves that we can move from simple RAG (Retrieval) to Agentic Workflows. In a production environment at Meta or similar scale, this reduces manual operational overhead by allowing the AI to handle "Level 1" troubleshooting and notification tasks autonomously, provided the tool schemas and quality gates are well-defined.



---



## Agent Trace & Quality Audit

In [15]:
import pandas as pd

trace_data = []
for i, msg in enumerate(messages):
    role = msg['role']
    content = msg['content']

    if isinstance(content, list):
        for block in content:
            # Fix: Using dot notation for Anthropic TextBlock/ToolUseBlock objects
            if hasattr(block, 'type'):
                if block.type == 'tool_use':
                    trace_data.append({"Turn": i, "Role": "Agent", "Action": f"CALL: {block.name}", "Details": str(block.input)})
                elif block.type == 'tool_result':
                    # tool_result is usually in a user message content list
                    trace_data.append({"Turn": i, "Role": "System", "Action": "RESULT", "Details": block.get('content') if isinstance(block, dict) else "Data Received"})
                elif block.type == 'text':
                    trace_data.append({"Turn": i, "Role": role.capitalize(), "Action": "Text", "Details": block.text[:100] + "..."})
    else:
        trace_data.append({"Turn": i, "Role": role.capitalize(), "Action": "Conversation", "Details": content[:100] + "..."})

df_trace = pd.DataFrame(trace_data)
print("### LOGISTICS AGENT: EXECUTION TRACE")
display(df_trace)

### LOGISTICS AGENT: EXECUTION TRACE


,Turn,Role,Action,Details
0,0,User,Conversation,Our shipment of sensors is delayed. Find the t...
1,1,Assistant,Text,I'll start by getting the order info for the s...
2,1,Agent,CALL: get_order_info,{'item': 'sensors'}
3,3,Assistant,Text,Got the tracking number **TRK-9904**. Now let ...
4,3,Agent,CALL: check_shipping,{'tracking_number': 'TRK-9904'}
5,5,Assistant,Text,"The ETA is **April 20, 2026 (Monday)**, which ..."
6,5,Agent,CALL: send_email,"{'recipient': 'floor manager', 'body': 'Hi, I ..."


**Insight**

- This test successfully validated the Claude 4.6 (Sonnet) model's ability to operate as a logic controller within a multi-step agentic loop.

- The pilot demonstrated that the model can resolve complex dependencies and adhere to business-critical "Quality Gates" without human intervention.



---



## Further Improvement


- This "Logistics Assistant" is a high-potential prototype, but it currently lacks the deterministic guardrails required for deployment. The next phase will focus on breaking the model's logic to identify exactly where autonomy turns into operational risk.



---

